In [1]:
# ════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & DATA LOADING
# ════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

print('=' * 55)
print('   DIAPILOT AI ENGINE v3.0 — Production Pipeline')
print('=' * 55)

df = pd.read_parquet('../data/processed/DiaPilot_Combined_Data.parquet')
print(f'Dataset loaded: {len(df):,} recipes')
print(f'Columns available: {list(df.columns)}')

   DIAPILOT AI ENGINE v3.0 — Production Pipeline
Dataset loaded: 117,662 recipes
Columns available: ['Recipe_Name', 'Ingredients', 'Calories', 'Carbs_g', 'Sugar_g', 'Protein_g', 'Fat_g', 'Source']


In [2]:
# ════════════════════════════════════════════════════════
# CELL 2 — ADA-ALIGNED MEDICAL ENGINE
# Fix #2: Sugar limits based on clinical tiers, not math
# ════════════════════════════════════════════════════════

def calculate_tdee(age, weight_kg, height_cm, gender, activity='sedentary'):
    """
    Mifflin-St Jeor Equation for Basal Metabolic Rate.
    Activity multipliers follow standard clinical practice.
    """
    if gender.lower() == 'male':
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    else:
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161

    multipliers = {
        'sedentary' : 1.2,
        'light'     : 1.375,
        'moderate'  : 1.55,
        'active'    : 1.725
    }
    return round(bmr * multipliers.get(activity, 1.2))


def get_dynamic_macros(tdee, leicester_score, on_medication=False,
                        current_glucose_high=False):
    """
    ADA 2025-aligned per-meal nutritional constraints.

    Carb tiers  (ADA Standard):
      Score <= 10  → 50% of calories from carbs  (Low risk / Prevention)
      Score <= 15  → 40% of calories from carbs  (Pre-diabetic)
      Score  > 15  → 30% of calories from carbs  (Diabetic / Management)

    Sugar limits  (ADA: minimize added sugar — no fixed per-meal cap):
      Prevention   → 15g per meal  (reasonable daily portion)
      Management   →  8g per meal  (strict minimization)

    Medication safety floor: 25g carbs minimum per meal
    to prevent hypoglycaemia — clinically non-negotiable.
    """
    if leicester_score <= 10:
        carb_pct       = 0.50
        sugar_limit    = 15.0
    elif leicester_score <= 15:
        carb_pct       = 0.40
        sugar_limit    = 10.0
    else:
        carb_pct       = 0.30
        sugar_limit    =  8.0

    # Emergency glucose override
    if current_glucose_high:
        carb_pct    -= 0.10
        sugar_limit  =  3.0

    per_meal_carbs_max = round(((tdee * carb_pct) / 4) / 3)
    per_meal_carbs_min = 25 if on_medication else 0

    # Hypoglycaemia safety: if calculated max is below the medication
    # floor, raise max to the floor so the window is always valid
    if on_medication and per_meal_carbs_max < 25:
        per_meal_carbs_max = 25

    return {
        'per_meal_calories'  : round(tdee / 3),
        'per_meal_max_carbs' : per_meal_carbs_max,
        'per_meal_min_carbs' : per_meal_carbs_min,
        'per_meal_sugar'     : sugar_limit
    }


# ── Quick sanity check ──────────────────────────────────
test_tdee   = calculate_tdee(age=45, weight_kg=85,
                              height_cm=175, gender='male')
test_macros = get_dynamic_macros(test_tdee,
                                  leicester_score=14,
                                  on_medication=True)

print('Medical Engine — ADA 2025 Aligned')
print('-' * 40)
print(f'  TDEE             : {test_tdee} kcal/day')
print(f'  Per-meal calories: {test_macros["per_meal_calories"]} kcal')
print(f'  Carb window      : {test_macros["per_meal_min_carbs"]}g '
      f'– {test_macros["per_meal_max_carbs"]}g')
print(f'  Sugar max        : {test_macros["per_meal_sugar"]}g')
print('Medical engine initialized.')

Medical Engine — ADA 2025 Aligned
----------------------------------------
  TDEE             : 2068 kcal/day
  Per-meal calories: 689 kcal
  Carb window      : 25g – 69g
  Sugar max        : 10.0g
Medical engine initialized.


In [3]:
# ════════════════════════════════════════════════════════
# CELL 3 — MEAL TYPE CLASSIFICATION
# Fix #4: Classify every recipe into Breakfast/Lunch/Dinner
# ════════════════════════════════════════════════════════

BREAKFAST_KW = [
    'egg', 'oat', 'oatmeal', 'paratha', 'halwa', 'dahi', 'yogurt',
    'pancake', 'toast', 'cereal', 'porridge', 'smoothie', 'muffin',
    'waffle', 'granola', 'scramble', 'omelette', 'crepe'
]

DINNER_KW = [
    'karahi', 'biryani', 'gosht', 'daal', 'curry', 'sabzi', 'stew',
    'roast', 'steak', 'bake', 'casserole', 'kebab', 'tikka', 'nihari',
    'korma', 'palak', 'keema', 'chops', 'grilled chicken', 'lamb'
]

def classify_meal_type(row):
    text = (str(row['Recipe_Name']) + ' ' +
            str(row['Ingredients'])).lower()

    b_score = sum(1 for kw in BREAKFAST_KW if kw in text)
    d_score = sum(1 for kw in DINNER_KW   if kw in text)

    if b_score > d_score:
        return 'Breakfast'
    elif d_score > 0:
        return 'Dinner'
    else:
        return 'Lunch'

df['Meal_Type'] = df.apply(classify_meal_type, axis=1)

# QID mapping for XGBoost Learning-to-Rank
# Fix #7: Meaningful QIDs — one per meal context, not one global ID
meal_qid_map = {'Breakfast': 1, 'Lunch': 2, 'Dinner': 3}
df['QID']      = df['Meal_Type'].map(meal_qid_map)

counts = df['Meal_Type'].value_counts()
print('Meal type classification complete')
print('-' * 40)
print(f'  Breakfast recipes: {counts.get("Breakfast", 0):,}')
print(f'  Lunch recipes    : {counts.get("Lunch", 0):,}')
print(f'  Dinner recipes   : {counts.get("Dinner", 0):,}')

Meal type classification complete
----------------------------------------
  Breakfast recipes: 28,429
  Lunch recipes    : 63,925
  Dinner recipes   : 25,308


In [4]:
# ════════════════════════════════════════════════════════
# CELL 4 — RELEVANCE LABEL ENGINEERING
# Fix #6: Normalize each component first so no single term
#         dominates and scores are never negative
# ════════════════════════════════════════════════════════

ADA_CARB_TARGET = 45.0   # ADA midpoint recommendation (g per meal)

# Normalize each score component independently to 0–1
# so the combined score is always between 0 and 1

# Carb proximity score: 1.0 = exactly 45g, falls off symmetrically
carb_score    = 1.0 - (abs(df['Carbs_g'] - ADA_CARB_TARGET) / ADA_CARB_TARGET).clip(0, 1)

# Protein reward: higher protein = better for blood sugar stability
protein_score = (df['Protein_g'] / (df['Protein_g'].max() + 1e-9)).clip(0, 1)

# Sugar penalty: lower sugar = better score
sugar_score   = (1.0 - df['Sugar_g'] / (df['Sugar_g'].max() + 1e-9)).clip(0, 1)

# Weighted combination — carbs matter most for diabetes management
raw_relevance = (
    carb_score    * 0.50 +
    protein_score * 0.30 +
    sugar_score   * 0.20
)   # Always 0.0 – 1.0, never negative

# Scale to integer labels 1–10 for XGBoost
scaler_label         = MinMaxScaler(feature_range=(1, 10))
df['Relevance_Label'] = (
    scaler_label
    .fit_transform(raw_relevance.values.reshape(-1, 1))
    .flatten()
    .astype(int)
)

print('Relevance label engineering complete')
print('-' * 40)
print(f'  Label range : {df["Relevance_Label"].min()} '
      f'– {df["Relevance_Label"].max()}')
print(f'  Label mean  : {df["Relevance_Label"].mean():.2f}')
print(f'  Raw score range: {raw_relevance.min():.4f} '
      f'– {raw_relevance.max():.4f}  (no negatives)')
print('Label distribution:')
print(df['Relevance_Label'].value_counts().sort_index().to_string())

Relevance label engineering complete
----------------------------------------
  Label range : 1 – 10
  Label mean  : 4.43
  Raw score range: 0.0232 – 0.8520  (no negatives)
Label distribution:
Relevance_Label
1       323
2      6714
3     34800
4     24906
5     19584
6     17003
7     11586
8      2708
9        37
10        1


In [5]:
# ════════════════════════════════════════════════════════
# CELL 5 — FEATURE EXTRACTION WITH CLINICAL WEIGHTING
# Fix #3: Nutrition features weighted 3x over TF-IDF
#         Carbs 2.5x, Sugar 2.0x within nutrition block
# ════════════════════════════════════════════════════════

# Text features — ingredient names (TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english', max_features=300)
X_text     = vectorizer.fit_transform(df['Ingredients'])

# Numerical features — macronutrients
# Column order: [Calories, Protein_g, Carbs_g, Sugar_g]
macro_cols  = ['Calories', 'Protein_g', 'Carbs_g', 'Sugar_g']
scaler_feat = MinMaxScaler()
X_num       = scaler_feat.fit_transform(df[macro_cols])

# Clinical importance weights
# Rationale: carbs and sugar have the highest glycaemic impact
clinical_weights = np.array([
    1.0,   # Calories    — moderate importance
    0.8,   # Protein_g   — least impact on blood glucose
    2.5,   # Carbs_g     — highest glycaemic impact
    2.0    # Sugar_g     — second highest glycaemic impact
])
X_num_weighted  = X_num * clinical_weights

# Give overall nutrition block 3x weight vs TF-IDF ingredient text
# Medical correctness > ingredient name similarity
X_num_weighted *= 3.0

# Combine into final feature matrix
X_full = hstack([X_text, X_num_weighted])

print('Feature extraction complete')
print('-' * 40)
print(f'  TF-IDF features     : {X_text.shape[1]}')
print(f'  Nutritional features: {X_num_weighted.shape[1]} '
      f'(weighted 3x)')
print(f'  Total feature dims  : {X_full.shape[1]}')
print(f'  Feature matrix size : {X_full.shape}')

Feature extraction complete
----------------------------------------
  TF-IDF features     : 300
  Nutritional features: 4 (weighted 3x)
  Total feature dims  : 304
  Feature matrix size : (117662, 304)


In [6]:
# ════════════════════════════════════════════════════════
# CELL 6 — XGBOOST RANKER TRAINING
# Fix #7: Sort by QID before fitting (XGBoost requirement)
# ════════════════════════════════════════════════════════

y_labels = df['Relevance_Label'].values
qids     = df['QID'].values

# XGBoost requires data sorted by QID in ascending order
sort_idx       = np.argsort(qids, kind='stable')
X_sorted       = X_full.tocsr()[sort_idx]
y_sorted       = y_labels[sort_idx]
qids_sorted    = qids[sort_idx]

print('Training XGBoost Ranker...')
print(f'  QID groups : {np.unique(qids_sorted)} '
      f'(Breakfast=1, Lunch=2, Dinner=3)')

ranker = xgb.XGBRanker(
    objective      = 'rank:ndcg',
    n_estimators   = 150,
    learning_rate  = 0.08,
    max_depth      = 6,
    subsample      = 0.8,
    random_state   = 42,
    verbosity      = 0
)

ranker.fit(X_sorted, y_sorted, qid=qids_sorted)

# Score the full dataset
df['AI_Rank_Score'] = ranker.predict(X_full)

print('XGBoost training complete')
print('-' * 40)
print(f'  Score range: {df["AI_Rank_Score"].min():.4f} '
      f'– {df["AI_Rank_Score"].max():.4f}')
print(f'  Score mean : {df["AI_Rank_Score"].mean():.4f}')

Training XGBoost Ranker...
  QID groups : [1 2 3] (Breakfast=1, Lunch=2, Dinner=3)
XGBoost training complete
----------------------------------------
  Score range: -5.3647 – 5.1027
  Score mean : -4.8211


In [7]:
# ════════════════════════════════════════════════════════
# CELL 7 — MEDICAL FILTERING
# Apply ADA-aligned constraints to create the safe pool
# ════════════════════════════════════════════════════════

# ── Patient profile (edit these values per patient) ─────
PATIENT_AGE       = 45
PATIENT_WEIGHT_KG = 85
PATIENT_HEIGHT_CM = 175
PATIENT_GENDER    = 'male'
PATIENT_ACTIVITY  = 'sedentary'
LEICESTER_SCORE   = 14
ON_MEDICATION     = True
GLUCOSE_HIGH      = False
# ────────────────────────────────────────────────────────

patient_tdee   = calculate_tdee(
    PATIENT_AGE, PATIENT_WEIGHT_KG,
    PATIENT_HEIGHT_CM, PATIENT_GENDER, PATIENT_ACTIVITY
)
patient_macros = get_dynamic_macros(
    patient_tdee, LEICESTER_SCORE,
    ON_MEDICATION, GLUCOSE_HIGH
)

print('Patient Profile')
print('-' * 40)
print(f'  TDEE             : {patient_tdee} kcal/day')
print(f'  Leicester score  : {LEICESTER_SCORE}')
print(f'  On medication    : {ON_MEDICATION}')
print(f'  Per-meal calories: <= {patient_macros["per_meal_calories"]} kcal')
print(f'  Carb window      : {patient_macros["per_meal_min_carbs"]}g '
      f'– {patient_macros["per_meal_max_carbs"]}g')
print(f'  Sugar max        : {patient_macros["per_meal_sugar"]}g')

# Apply medical filter
safe_df = df[
    (df['Calories'] <= patient_macros['per_meal_calories']) &
    (df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (df['Sugar_g']  <= patient_macros['per_meal_sugar'])
].copy().reset_index(drop=True)

print(f'\nMedically safe recipes: {len(safe_df):,}')

safe_counts = safe_df['Meal_Type'].value_counts()
print(f'  Breakfast pool : {safe_counts.get("Breakfast", 0):,}')
print(f'  Lunch pool     : {safe_counts.get("Lunch", 0):,}')
print(f'  Dinner pool    : {safe_counts.get("Dinner", 0):,}')

if len(safe_df) < 90:
    print('\nWARNING: Fewer than 90 safe recipes found.')
    print('Consider relaxing clinical thresholds or using Gemini localization.')

Patient Profile
----------------------------------------
  TDEE             : 2068 kcal/day
  Leicester score  : 14
  On medication    : True
  Per-meal calories: <= 689 kcal
  Carb window      : 25g – 69g
  Sugar max        : 10.0g

Medically safe recipes: 21,763
  Breakfast pool : 4,325
  Lunch pool     : 13,228
  Dinner pool    : 4,210


In [8]:
# ════════════════════════════════════════════════════════
# CELL 8 — MMR DIVERSITY SELECTION
# Fix #5: Multi-macro similarity (not carbs only)
# Fix #5: Applied to ALL meal types including Breakfast
# ════════════════════════════════════════════════════════

def apply_mmr(candidate_df, top_k, lambda_param=0.7):
    """
    Maximum Marginal Relevance selection.

    Balances:
      Relevance  (AI_Rank_Score from XGBoost)  — lambda weight
      Diversity  (nutritional dissimilarity)    — (1-lambda) weight

    Similarity is computed across all 4 macros so that
    two meals are only considered similar if they match
    on Calories, Carbs, Sugar AND Protein simultaneously.

    Parameters
    ----------
    candidate_df  : DataFrame filtered to one meal type
    top_k         : number of meals to select (e.g. 30 for 30 days)
    lambda_param  : 0 = max diversity, 1 = max relevance
                    0.7 = good balance for diet planning
    """
    if len(candidate_df) == 0:
        return pd.DataFrame()

    # If we have fewer candidates than needed, return all
    if len(candidate_df) <= top_k:
        return candidate_df.reset_index(drop=True)

    # Normalize macro columns to 0-1 for fair distance calculation
    macro_cols = ['Calories', 'Carbs_g', 'Sugar_g', 'Protein_g']
    macro_vals = candidate_df[macro_cols].values.astype(float)
    col_min    = macro_vals.min(axis=0)
    col_max    = macro_vals.max(axis=0)
    col_range  = np.where(col_max - col_min > 0, col_max - col_min, 1.0)
    macro_norm = (macro_vals - col_min) / col_range  # Each column 0-1

    scores     = candidate_df['AI_Rank_Score'].values
    remaining  = list(range(len(candidate_df)))
    selected   = []

    # Always start with the highest-ranked meal
    best_start = int(np.argmax(scores))
    selected.append(best_start)
    remaining.remove(best_start)

    while len(selected) < top_k and remaining:
        best_mmr = -float('inf')
        best_idx = None

        for i in remaining:
            relevance = scores[i]

            # Similarity = 1 / (1 + euclidean distance across all 4 macros)
            # Lower distance → higher similarity → higher diversity penalty
            sims = [
                1.0 / (1.0 + np.linalg.norm(macro_norm[i] - macro_norm[s]))
                for s in selected
            ]
            max_sim = max(sims)

            # MMR = λ × Relevance − (1−λ) × MaxSimilarity
            mmr = lambda_param * relevance - (1 - lambda_param) * max_sim

            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i

        selected.append(best_idx)
        remaining.remove(best_idx)

    return candidate_df.iloc[selected].reset_index(drop=True)


print('MMR function defined.')
print('Similarity computed across: Calories, Carbs, Sugar, Protein')
print('Applied to: Breakfast + Lunch + Dinner (all meal types)')

MMR function defined.
Similarity computed across: Calories, Carbs, Sugar, Protein
Applied to: Breakfast + Lunch + Dinner (all meal types)


In [9]:
# ════════════════════════════════════════════════════════
# CELL 9 — GENERATE COMPLETE 30-DAY MEAL PLAN
# Fix: 90 total meals (30 days × 3 slots)
# Fix: Breakfast included with its own MMR-selected pool
# Fix: Fallback cycling when pool < 30 recipes
# ════════════════════════════════════════════════════════

DAYS_IN_PLAN = 30
MEAL_SLOTS   = ['Breakfast', 'Lunch', 'Dinner']

plan_rows    = []
pool_summary = {}

for meal_type in MEAL_SLOTS:
    # Get medically safe pool for this meal type
    pool = safe_df[safe_df['Meal_Type'] == meal_type].copy()

    # Fallback: if a meal type has very few recipes,
    # borrow from the full safe pool
    if len(pool) < 10:
        print(f'  WARNING: Only {len(pool)} {meal_type} recipes '
              f'found — using full safe pool as fallback.')
        pool = safe_df.copy()

    # Run MMR to get maximally diverse set of 30 meals
    selected = apply_mmr(pool, top_k=DAYS_IN_PLAN, lambda_param=0.7)
    pool_summary[meal_type] = len(selected)

    # Build one row per day for this meal slot
    for day_idx in range(DAYS_IN_PLAN):
        # Cycle if MMR returned fewer than 30 unique meals
        row = selected.iloc[day_idx % len(selected)]

        plan_rows.append({
            'Day'       : f'Day {day_idx + 1}',
            'Day_Num'   : day_idx + 1,
            'Meal'      : meal_type,
            'Recipe'    : row['Recipe_Name'],
            'Calories'  : round(row['Calories'],  1),
            'Carbs_g'   : round(row['Carbs_g'],   1),
            'Sugar_g'   : round(row['Sugar_g'],   1),
            'Protein_g' : round(row['Protein_g'], 1),
            'AI_Score'  : round(row['AI_Rank_Score'], 4)
        })

# Assemble and sort plan by day then meal order
plan_df = pd.DataFrame(plan_rows)
meal_order = {'Breakfast': 0, 'Lunch': 1, 'Dinner': 2}
plan_df['Meal_Order'] = plan_df['Meal'].map(meal_order)
plan_df = (
    plan_df
    .sort_values(['Day_Num', 'Meal_Order'])
    .drop(columns=['Day_Num', 'Meal_Order'])
    .reset_index(drop=True)
)

print('30-Day Meal Plan Generated')
print('=' * 55)
print(f'  Total meals      : {len(plan_df)}  (30 days × 3 slots)')
for mt, count in pool_summary.items():
    print(f'  {mt:12s} unique meals selected: {count}')

print('\nSample — First 9 meals (Days 1-3):')
print('-' * 55)
display_cols = ['Day', 'Meal', 'Recipe', 'Calories', 'Carbs_g', 'Sugar_g']
print(plan_df[display_cols].head(9).to_string(index=False))

30-Day Meal Plan Generated
  Total meals      : 90  (30 days × 3 slots)
  Breakfast    unique meals selected: 30
  Lunch        unique meals selected: 30
  Dinner       unique meals selected: 30

Sample — First 9 meals (Days 1-3):
-------------------------------------------------------
  Day      Meal                                        Recipe  Calories  Carbs_g  Sugar_g
Day 1 Breakfast                          perfect fish   chips     542.6     46.8      0.5
Day 1     Lunch                    hamburger and kidney beans     587.4     46.8      1.0
Day 1    Dinner             hot roast beef sandwiches   gravy     666.6     44.0      2.5
Day 2 Breakfast               linda s new england fried clams     674.5     41.2      0.5
Day 2     Lunch       chicken fettuccine with ricotta   basil     671.0     44.0      2.5
Day 2    Dinner          pork chop  onion  and rice casserole     646.9     46.8      0.5
Day 3 Breakfast     pheasant nuggets deep fried and delicious     521.5     41.2   

In [10]:
# ════════════════════════════════════════════════════════
# CELL 10 — MODEL EVALUATION
# Evaluate diversity, medical compliance, and variety
# ════════════════════════════════════════════════════════

print('=' * 55)
print('   DIAPILOT MODEL EVALUATION REPORT')
print('=' * 55)

# ── 1. Medical Compliance ───────────────────────────────
compliant = plan_df[
    (plan_df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (plan_df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (plan_df['Sugar_g']  <= patient_macros['per_meal_sugar']) &
    (plan_df['Calories'] <= patient_macros['per_meal_calories'])
]
compliance_pct = (len(compliant) / len(plan_df)) * 100

print(f'\n1. Medical Compliance')
print(f'   ADA-safe meals   : {len(compliant)} / {len(plan_df)}')
print(f'   Compliance rate  : {compliance_pct:.1f}%')

# ── 2. Nutritional Stats ────────────────────────────────
print(f'\n2. Nutritional Profile (plan average per meal)')
print(f'   Avg Calories : {plan_df["Calories"].mean():.1f} kcal '
      f'(limit: {patient_macros["per_meal_calories"]})')
print(f'   Avg Carbs    : {plan_df["Carbs_g"].mean():.1f}g '
      f'(window: {patient_macros["per_meal_min_carbs"]}'
      f'-{patient_macros["per_meal_max_carbs"]}g)')
print(f'   Avg Sugar    : {plan_df["Sugar_g"].mean():.1f}g '
      f'(limit: {patient_macros["per_meal_sugar"]}g)')
print(f'   Avg Protein  : {plan_df["Protein_g"].mean():.1f}g')

# ── 3. Variety Score ────────────────────────────────────
total_meals  = len(plan_df)
unique_meals = plan_df['Recipe'].nunique()
variety_pct  = (unique_meals / total_meals) * 100

print(f'\n3. Variety Score')
print(f'   Unique recipes : {unique_meals} / {total_meals}')
print(f'   Variety score  : {variety_pct:.1f}%  '
      f'(100% = no repeated meals)')

# ── 4. Per-Slot Variety ─────────────────────────────────
print(f'\n4. Per-Slot Variety')
for mt in MEAL_SLOTS:
    slot_df   = plan_df[plan_df['Meal'] == mt]
    slot_uniq = slot_df['Recipe'].nunique()
    print(f'   {mt:12s}: {slot_uniq} unique / 30 days')

# ── 5. Carb Distribution Check ──────────────────────────
print(f'\n5. Carb Distribution (meals in ADA 45-60g range)')
in_ada_range = plan_df[
    (plan_df['Carbs_g'] >= 45) & (plan_df['Carbs_g'] <= 60)
]
print(f'   In ADA 45-60g range: {len(in_ada_range)} / {len(plan_df)} meals')

print('\n' + '=' * 55)
print('   Evaluation complete. Ready for Gemini localization.')
print('=' * 55)

   DIAPILOT MODEL EVALUATION REPORT

1. Medical Compliance
   ADA-safe meals   : 90 / 90
   Compliance rate  : 100.0%

2. Nutritional Profile (plan average per meal)
   Avg Calories : 579.5 kcal (limit: 689)
   Avg Carbs    : 43.7g (window: 25-69g)
   Avg Sugar    : 2.6g (limit: 10.0g)
   Avg Protein  : 43.6g

3. Variety Score
   Unique recipes : 90 / 90
   Variety score  : 100.0%  (100% = no repeated meals)

4. Per-Slot Variety
   Breakfast   : 30 unique / 30 days
   Lunch       : 30 unique / 30 days
   Dinner      : 30 unique / 30 days

5. Carb Distribution (meals in ADA 45-60g range)
   In ADA 45-60g range: 23 / 90 meals

   Evaluation complete. Ready for Gemini localization.


In [11]:
# ════════════════════════════════════════════════════════
# CELL 11 — PDF REPORT GENERATION
# Clean, properly formatted output with all 90 meals
# ════════════════════════════════════════════════════════

from fpdf import FPDF
import os

# Helper: strip any character outside latin-1 range
def to_latin1(text):
    return str(text).encode('latin-1', errors='replace').decode('latin-1')

class DiapilotPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 13)
        self.set_text_color(29, 158, 117)   # Diapilot teal
        self.cell(0, 10, 'DIAPILOT - 30-Day Medical Diet Plan', ln=True, align='C')
        self.set_font('Arial', 'I', 9)
        self.set_text_color(100, 100, 100)
        self.cell(0, 6, 'XGBoost Ranker + MMR Diversity | ADA 2025 Guidelines',
                  ln=True, align='C')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10,
                  f'Page {self.page_no()} | Generated by Diapilot AI Engine v3.0',
                  align='C')


pdf = DiapilotPDF()
pdf.set_auto_page_break(auto=True, margin=20)
pdf.add_page()

# ── Patient summary box ─────────────────────────────────
pdf.set_font('Arial', 'B', 10)
pdf.set_fill_color(225, 245, 238)   # Light teal
pdf.set_text_color(8, 80, 65)
pdf.cell(0, 8,
         f'Patient Profile  |  TDEE: {patient_tdee} kcal  |  '
         f'Leicester Score: {LEICESTER_SCORE}  |  '
         f'Medication: {"Yes" if ON_MEDICATION else "No"}  |  '
         f'Carbs: {patient_macros["per_meal_min_carbs"]}'
         f'-{patient_macros["per_meal_max_carbs"]}g  |  '
         f'Sugar: max {patient_macros["per_meal_sugar"]}g',
         border=1, ln=True, fill=True, align='C')
pdf.ln(4)

# ── Table header ────────────────────────────────────────
col_widths = [18, 25, 75, 20, 18, 18, 16]
headers    = ['Day', 'Meal', 'Recipe Name', 'Cal', 'Carbs', 'Sugar', 'Score']

pdf.set_font('Arial', 'B', 9)
pdf.set_fill_color(29, 158, 117)
pdf.set_text_color(255, 255, 255)
for h, w in zip(headers, col_widths):
    pdf.cell(w, 8, h, border=1, align='C', fill=True)
pdf.ln()

# ── Table rows ──────────────────────────────────────────
meal_colors = {
    'Breakfast': (255, 253, 240),   # Warm yellow
    'Lunch'    : (240, 248, 255),   # Light blue
    'Dinner'   : (245, 240, 255),   # Light purple
}

pdf.set_font('Arial', '', 8)
current_day = ''

for _, row in plan_df.iterrows():
    # Shade alternating days lightly
    r, g, b = meal_colors[row['Meal']]
    pdf.set_fill_color(r, g, b)
    pdf.set_text_color(30, 30, 30)

    day_label = row['Day'] if row['Day'] != current_day else ''
    current_day = row['Day']

    clean_name = to_latin1(row['Recipe'])[:38].title()

    pdf.cell(col_widths[0], 7, day_label,             border=1, align='C', fill=True)
    pdf.cell(col_widths[1], 7, row['Meal'],            border=1, align='C', fill=True)
    pdf.cell(col_widths[2], 7, clean_name,             border=1,            fill=True)
    pdf.cell(col_widths[3], 7, f"{row['Calories']}" ,  border=1, align='C', fill=True)
    pdf.cell(col_widths[4], 7, f"{row['Carbs_g']}g",   border=1, align='C', fill=True)
    pdf.cell(col_widths[5], 7, f"{row['Sugar_g']}g",   border=1, align='C', fill=True)
    pdf.cell(col_widths[6], 7, f"{row['AI_Score']:.2f}",border=1, align='C', fill=True)
    pdf.ln()

# ── Save ────────────────────────────────────────────────
os.makedirs('../reports', exist_ok=True)
output_path = '../reports/Diapilot_v3_Production_Plan.pdf'
pdf.output(output_path)

print(f'PDF generated: {output_path}')
print(f'Total meals in report: {len(plan_df)}')
print('Ready for Gemini localization pass.')

PDF generated: ../reports/Diapilot_v3_Production_Plan.pdf
Total meals in report: 90
Ready for Gemini localization pass.


In [12]:
# ════════════════════════════════════════════════════════
# CELL 12 — SAVE MODEL ARTIFACTS
# Save trained model and preprocessors for Flask API use
# ════════════════════════════════════════════════════════

import joblib

os.makedirs('../models', exist_ok=True)

joblib.dump(ranker,       '../models/diapilot_xgb_ranker.pkl')
joblib.dump(vectorizer,   '../models/diapilot_tfidf_vectorizer.pkl')
joblib.dump(scaler_feat,  '../models/diapilot_macro_scaler.pkl')

# Save the plan as CSV for React Native / Appwrite upload
plan_df.to_csv('../reports/Diapilot_v3_30Day_Plan.csv', index=False)

print('Model artifacts saved')
print('-' * 40)
print('  ../models/diapilot_xgb_ranker.pkl')
print('  ../models/diapilot_tfidf_vectorizer.pkl')
print('  ../models/diapilot_macro_scaler.pkl')
print('  ../reports/Diapilot_v3_30Day_Plan.csv')
print('\nDIAPILOT v3.0 pipeline complete.')

Model artifacts saved
----------------------------------------
  ../models/diapilot_xgb_ranker.pkl
  ../models/diapilot_tfidf_vectorizer.pkl
  ../models/diapilot_macro_scaler.pkl
  ../reports/Diapilot_v3_30Day_Plan.csv

DIAPILOT v3.0 pipeline complete.
